# T14: RAG als Subagent

In T13 hatte ein Agent das Retrieval-Tool. Hier wird der RAG-Agent selbst zum Tool eines **übergeordneten Agenten**. Der übergeordnete Agent führt die Konversation und entscheidet, welches Tool er aufruft; der RAG-Subagent übernimmt Retrieval und Belege. Der übergeordnete Agent sieht nie Chunks, nur die Antwort des Subagenten.

```
User ──► Parent agent ──► rag_agent tool ──► RAG subagent ──► retriever ──► vector store
              ├─────────► sum_two_numbers
              └─────────► answer directly
```

Code, Prompts und Dokumente sind auf Englisch (Begründung in T01).

## Setup

Zuerst T01 ausführen: Dort entsteht der Vector Store unter `backend/db/data_vs.db`. Ist der Store leer, baut `load_or_build_vector_store()` ihn aus allen Dokumenten auf; das dauert einige Minuten.

In [ ]:
import sys

from loguru import logger

from conversational_toolkit.agents.tool_agent import ToolAgent
from conversational_toolkit.llms.base import LLMMessage, MessageContent, Roles
from conversational_toolkit.llms.openai import OpenAILLM
from conversational_toolkit.retriever.vectorstore_retriever import VectorStoreRetriever

from ki_kompass_rag.baseline_rag import RETRIEVER_TOP_K, ask, load_or_build_vector_store
from ki_kompass_rag.tool_agents import (
    RAGAgentAsTool,
    RetrieveRelevantChunks,
    SumTwoNumbers,
)

logger.remove()
logger.add(sys.stderr, level="WARNING")

vector_store, embedding_model = await load_or_build_vector_store()
retriever = VectorStoreRetriever(embedding_model, vector_store, top_k=RETRIEVER_TOP_K)

## RAG-Subagent

Derselbe Agent wie in T13, aber er wird nur für fachliche Fragen aufgerufen; sein Prompt macht das Retrieval deshalb zur Pflicht.

`RAG_PROMPT` übersetzt:

> Du beantwortest Fragen zur PrimePack AG. Rufe vor jeder Antwort retrieve_relevant_chunks auf. Stütze dich nur auf die gefundenen Chunks und nenne ihre Quelldateien. Antworte kurz.

In [ ]:
RAG_PROMPT = "You answer questions about PrimePack AG. Always call retrieve_relevant_chunks before answering. Rely only on the retrieved chunks and cite their source files. Answer briefly."

rag_llm = OpenAILLM(tools=[RetrieveRelevantChunks(retriever)], tool_choice="auto")
rag_agent = ToolAgent(system_prompt=RAG_PROMPT, llm=rag_llm, max_steps=5)

## Übergeordneter Agent

`RAGAgentAsTool` verpackt den Subagenten als Tool: `call` führt `rag_agent.answer` aus und gibt den Antworttext als `result` und die Quellen des Subagenten als `_sources` zurück. Der Subagent erhält keinen Verlauf, der übergeordnete Agent muss also eine in sich verständliche Frage schicken.

Der übergeordnete Agent hat zusätzlich `SumTwoNumbers` und wählt deshalb pro Anfrage: direkt antworten, Zahlen addieren, den Subagenten fragen oder beides kombinieren.

`PARENT_PROMPT` übersetzt:

> Du bist ein Assistent der PrimePack AG. Antworte kurz. Verwende rag_agent für Fragen zu Produkten, Lieferanten oder Dokumenten von PrimePack, mit einer Frage pro Produkt, und übernimm die Quelldateien, die er nennt. Verwende sum_two_numbers für Additionen.

In [ ]:
PARENT_PROMPT = "You are an assistant for PrimePack AG. Answer briefly. Use rag_agent for questions about PrimePack products, suppliers or documents, with one question per product, and keep the source files it cites. Use sum_two_numbers for additions."

parent_llm = OpenAILLM(
    tools=[RAGAgentAsTool(rag_agent), SumTwoNumbers()], tool_choice="auto"
)
parent_agent = ToolAgent(system_prompt=PARENT_PROMPT, llm=parent_llm, max_steps=5)

## Konversation

`ask` zeigt die Quelldateien aus allen Aufrufen des Subagenten; `Sources: none` heisst, der übergeordnete Agent hat den Subagenten nicht aufgerufen. Die dritte Frage braucht sowohl den Subagenten als auch das Additions-Tool.

In [ ]:
history: list[LLMMessage] = []


async def chat(agent: ToolAgent, query: str) -> None:
    answer = await ask(agent, query, history)
    user_message = LLMMessage(
        role=Roles.USER, content=[MessageContent(type="text", text=query)]
    )
    history.extend([user_message, answer])

In [ ]:
await chat(
    parent_agent, "What is Einstein's theory of relativity? Answer in 2-3 sentences."
)

In [ ]:
await chat(
    parent_agent, "Which pallets in our portfolio have a third-party verified EPD?"
)

In [ ]:
await chat(
    parent_agent,
    "What is the total empty weight of a CPR wood pallet and a CPR recycled plastic pallet?",
)

In [ ]:
await chat(parent_agent, "Summarize our conversation, one line per message.")